# CNN LeNet-5 – MNIST / Fashion MNIST / Medical MNIST

**So sánh 3 tầng:**
| Variant | Mô tả |
|---|---|
| **Baseline** | LeNet-5 gốc – Tanh + AvgPool |
| **Custom** | Riêng từng dataset – filter/depth được thiết kế theo độ khó |
| **Improved** | Chung cho cả 3 – ReLU + BN + Dropout + MaxPool |


## 0. Clone repo & Setup path

In [ ]:
import os, sys

REPO_URL = 'https://github.com/PhuongThao-2005/MNIST-CNNLeNet5.git'
REPO_DIR = '/kaggle/working/MNIST-CNNLeNet5'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Files:', os.listdir(REPO_DIR))

## 1. Import

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import os

from config  import DEVICE, BATCH_SIZE, EPOCHS
from model   import LeNet5, LeNet5Improved, LeNet5_Handwritten, LeNet5_Fashion, LeNet5_Medical
from dataset import get_handwritten_mnist, get_fashion_mnist, get_medical_mnist
from train   import train_model

print(f'Device: {DEVICE} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS}')

## 2. Load Dataset

In [ ]:
# Load datasets
handwritten_train, handwritten_test, handwritten_classes = get_handwritten_mnist(BATCH_SIZE)
fashion_train, fashion_test, fashion_classes = get_fashion_mnist(BATCH_SIZE)
medical_train, medical_test, medical_classes = get_medical_mnist(BATCH_SIZE)

datasets = {
    "handwritten": {
        "train": handwritten_train,
        "test": handwritten_test,
        "classes": handwritten_classes,
        "num_classes": len(handwritten_classes),
    },
    "fashion": {
        "train": fashion_train,
        "test": fashion_test,
        "classes": fashion_classes,
        "num_classes": len(fashion_classes),
    },
    "medical": {
        "train": medical_train,
        "test": medical_test,
        "classes": medical_classes,
        "num_classes": len(medical_classes),
    },
}

for name, info in datasets.items():
    n_train = len(info['train'].dataset)
    n_test  = len(info['test'].dataset)
    print(f'{name.upper():10s}: train={n_train:>6,}  test={n_test:>5,}  classes={info["num_classes"]}')
    print(f'  Labels: {info["classes"]}')

## 3. Visualise mẫu dữ liệu

In [ ]:
def show_samples(loader, class_names, title, n=10):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(16, 2))
    fig.suptitle(title, fontsize=12)
    for i in range(n):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].set_title(class_names[labels[i]], fontsize=7)
        axes[i].axis('off')
    plt.tight_layout(); plt.show()

show_samples(datasets['handwritten']['train'], datasets['handwritten']['classes'], 'Handwritten MNIST')
show_samples(datasets['fashion']['train'], datasets['fashion']['classes'], 'Fashion MNIST')
show_samples(datasets['medical']['train'], datasets['medical']['classes'], 'Medical MNIST')

## 4. So sánh kiến trúc 5 model

| Model          | Filters       | Layers     | FC          | Dropout   | Lý do                                         |
| -------------- | ------------- | ---------- | ----------- | --------- | --------------------------------------------- |
| Baseline       | 6→16          | 2 Conv     | 120→84      | Không     | Chuẩn gốc LeNet                               |
| Improved       | 6→16          | 2 Conv     | 120→84      | 0.4       | Thêm BN + ReLU + ổn định train                |
| MNIST Custom   | **6→12**      | 2 Conv     | **84**      | **0.3**   | MNIST đơn giản → giảm capacity, tránh overfit |
| Fashion Custom | **32→64→128** | **3 Conv** | **512→256** | **0.5×2** | Texture phức tạp → cần model lớn hơn          |
| Medical Custom | **4→8**       | 2 Conv     | **32**      | Không     | Dataset dễ phân biệt → model nhỏ đủ           |


In [ ]:
models_info = [
    (LeNet5,          10, 'LeNet5 Baseline'),
    (LeNet5Improved,  10, 'LeNet5 Improved (chung)'),
    (LeNet5_Handwritten,  10, 'LeNet5_Handwritten  (riêng - nhẹ + Dropout 0.3)'),
    (LeNet5_Fashion,  10, 'LeNet5_Fashion  (riêng - 3 conv)'),
    (LeNet5_Medical,   6, 'LeNet5_Medical  (riêng - deeper)'),
]

dummy = torch.zeros(2, 1, 32, 32)
print(f'{"Model":<38} {"Output":>14} {"Params":>12}')
print('-' * 66)
for cls, nc, name in models_info:
    m = cls(num_classes=nc)
    out = m(dummy)
    params = sum(p.numel() for p in m.parameters())
    print(f'{name:<38} {str(out.shape):>14} {params:>12,}')

## 5. Train Baseline trên cả 3 dataset

In [ ]:
# Train baseline cho tất cả datasets
results = {}

for ds_name in ['handwritten', 'fashion', 'medical']:
    ds_info = datasets[ds_name]
    model_class = LeNet5  # baseline
    model, history = train_model(
        train_loader=ds_info['train'],
        test_loader=ds_info['test'],
        dataset=ds_name,
        variant='baseline',
        num_classes=ds_info['num_classes'],
        epochs=20,  # hoặc từ config
        lr=0.001
    )
    results[ds_name] = {'baseline': {'model': model, 'history': history}}
    
    # Lưu model
    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), f'models/{ds_name}_baseline.pth')
    print(f'Saved {ds_name}_baseline.pth')

## 6. Train Custom cho từng dataset

In [ ]:
# Train custom cho từng dataset
custom_models = {
    'handwritten': LeNet5_Handwritten,
    'fashion': LeNet5_Fashion,
    'medical': LeNet5_Medical
}

for ds_name in ['handwritten', 'fashion', 'medical']:
    ds_info = datasets[ds_name]
    model_class = custom_models[ds_name]
    model, history = train_model(
        train_loader=ds_info['train'],
        test_loader=ds_info['test'],
        dataset=ds_name,
        variant='custom',
        num_classes=ds_info['num_classes'],
        epochs=20,
        lr=0.001
    )
    results[ds_name]['custom'] = {'model': model, 'history': history}
    
    # Lưu model
    torch.save(model.state_dict(), f'models/{ds_name}_custom.pth')
    print(f'Saved {ds_name}_custom.pth')

## 7. Train Improved trên cả 3 dataset

In [ ]:
# Train improved cho tất cả datasets
for ds_name in ['handwritten', 'fashion', 'medical']:
    ds_info = datasets[ds_name]
    model_class = LeNet5Improved
    model, history = train_model(
        train_loader=ds_info['train'],
        test_loader=ds_info['test'],
        dataset=ds_name,
        variant='improved',
        num_classes=ds_info['num_classes'],
        epochs=20,
        lr=0.001
    )
    results[ds_name]['improved'] = {'model': model, 'history': history}
    
    # Lưu model
    torch.save(model.state_dict(), f'models/{ds_name}_improved.pth')
    print(f'Saved {ds_name}_improved.pth')

In [ ]:
# Bảng tổng hợp Best Test Accuracy
def print_summary(results):
    VARIANTS = ['baseline', 'custom', 'improved']
    header = f"{'Dataset':<12}" + "".join(f"{v.upper():>12}" for v in VARIANTS)
    sep    = "=" * (12 + 12 * len(VARIANTS))
    print(f"\n{sep}\n{header}\n{'-' * (12 + 12 * len(VARIANTS))}")
    for ds_name, ds_results in results.items():
        row = f"{ds_name.upper():<12}"
        for variant in VARIANTS:
            best = max(ds_results[variant]["history"]["test_acc"])
            row += f"{best:>12.4f}"
        print(row)
    print(sep)

print_summary(results)

## 7. So sánh Accuracy Curve – 3 variant trên mỗi dataset

In [ ]:
COLORS = {'baseline': '#e74c3c', 'improved': '#3498db', 'custom': '#2ecc71'}
VARIANTS = ['baseline', 'custom', 'improved']

def plot_all_variants(results, ds_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{ds_name.upper()} – Baseline vs Custom vs Improved', fontsize=13)

    for variant in VARIANTS:
        h = results[ds_name][variant]['history']
        ep = range(1, len(h['loss']) + 1)
        c  = COLORS[variant]
        axes[0].plot(ep, h['loss'],      color=c, label=variant)
        axes[1].plot(ep, h['train_acc'], color=c, label=variant)
        axes[2].plot(ep, h['test_acc'],  color=c, label=variant)

    for ax, title in zip(axes, ['Loss', 'Train Accuracy', 'Test Accuracy']):
        ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')

    plt.tight_layout(); plt.show()

for ds in ['handwritten', 'fashion', 'medical']:
    plot_all_variants(results, ds)

## 8. Confusion Matrix – Custom model (model tốt nhất)

In [ ]:
def get_predictions(model, loader):
    model.eval()
    preds_all, labels_all, images_all = [], [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            p    = model(X).argmax(dim=1)
            preds_all.extend(p.cpu().numpy())
            labels_all.extend(y.cpu().numpy())
            images_all.extend(X.cpu())
    return preds_all, labels_all, images_all


def plot_confusion_matrix(preds, labels, class_names, title):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.tight_layout(); plt.show()


# Đánh giá trên Custom model
for ds_name in ['handwritten', 'fashion', 'medical']:
    model   = results[ds_name]['custom']['model']
    loader  = datasets[ds_name]['test']
    classes = datasets[ds_name]['classes']

    preds, labels, images = get_predictions(model, loader)
    plot_confusion_matrix(preds, labels, classes,
                          f'{ds_name.upper()} – Confusion Matrix (Custom)')
    print(classification_report(labels, preds, target_names=classes))

    datasets[ds_name]['preds']  = preds
    datasets[ds_name]['labels'] = labels

## 9. Ảnh dự đoán sai – Custom model

In [ ]:
def show_wrong_predictions(images, preds, labels, class_names, title, num_images=9):
    wrong = [i for i in range(len(preds)) if preds[i] != labels[i]]
    if not wrong:
        print('Không có dự đoán sai!'); return
    n   = min(num_images, len(wrong))
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(f'{title} – Wrong ({len(wrong):,} total)', fontsize=12)
    for i, ax in enumerate(axes.flatten()):
        if i >= n:
            ax.axis('off'); continue
        idx = wrong[i]
        ax.imshow(images[idx].squeeze(), cmap='gray')
        ax.set_title(
            f'True : {class_names[labels[idx]]}\nPred : {class_names[preds[idx]]}',
            fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

for ds_name in ['handwritten', 'fashion', 'medical']:
    show_wrong_predictions(
        datasets[ds_name]['images'], datasets[ds_name]['preds'],
        datasets[ds_name]['labels'], datasets[ds_name]['classes'],
        ds_name.upper()
    )

## 10. Phân tích & Kết luận

### Nhận xét thiết kế

**MNIST Handwritten (Custom – LeNet5_MNIST):**


**Fashion MNIST (Custom – LeNet5_Fashion):**

**Medical MNIST (Custom – LeNet5_Medical):**